# Attention CNN, trained from scratch

Model 2 of 4. Ten conv layers (5 stages) down to a 16x16 feature map. Instead of averaging the
feature map, a 1x1 conv scores every cell, a softmax turns the scores into weights, and the class is predicted from
the weighted features. The attention is learned from the class labels only (no bounding boxes).

It uses exactly the same data, split, batches, augmentation, optimizer and seeds as `01_simple_cnn.ipynb`;
only the network differs.

* Input: the whole image, only resized to 512x512. Target: the class id from the `.txt` label file.
* Scene-wise split, identical for all models: 116 / 25 / 25 scenes (1740 / 375 / 375 images).
* One full training run per seed; the test accuracy is reported as mean +- std over the seeds.
* All code lives in the repository folder `src/`; this notebook only calls it.
* Outputs: `outputs/<model>/weights/`, `figures/` and `results/`, copied to Google Drive at the end.

**Colab:** Runtime -> Change runtime type -> **GPU**, then Run all.

## 1. Setup: code and data

In [ ]:
REPO_URL = "https://github.com/Ahmed-AlTahan/CNN-LIME.git"
BRANCH = "main"
REPO_DIR = "/content/thesis"
DATA_URL = "https://drive.google.com/file/d/1OfQmMsNR5DDLbZUmLL2DGuHPIGFxnqy6/view?usp=drive_link"

import os, sys
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git checkout -q {BRANCH} && git pull -q
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

if not os.path.isdir("data/15workpieces_same_position/all_images"):
    !pip -q install gdown
    !gdown -q --fuzzy "{DATA_URL}" -O data.zip && mkdir -p data && unzip -q data.zip -d data && rm data.zip
!nvidia-smi -L

## 2. Settings
The recipe comes from `config.RECIPES`; change it here for an experiment.

In [ ]:
import config, data, engine, report

recipe = config.RECIPES["attention_cnn"]
# Change settings here if needed, e.g.:
# recipe = config.replace(recipe, seeds=[1, 2, 3, 4, 5])
out_dir = report.output_dir(recipe.model)
print(recipe)
print("outputs ->", out_dir)

## 3. Load data
Every image is decoded once and kept in memory (a few minutes).

In [ ]:
train_ds, val_ds, test_ds = data.load_splits()
val_loader = data.make_eval_loader(val_ds, recipe.batch_size)
test_loader = data.make_eval_loader(test_ds, recipe.batch_size)

## 4. Check the inputs
Augmented training images, exactly as the network sees them.

In [ ]:
report.plot_inputs(train_ds, recipe, out_dir)

## 5. Sanity check
A fresh model must overfit one scene (15 images) to ~100%. If it cannot, the pipeline is broken.

In [ ]:
engine.sanity_check(recipe, train_ds)

## 6. Training
One run per seed. Re-running this cell gives the same results.

In [ ]:
runs = engine.train_all(recipe, train_ds, val_loader, out_dir)
report.plot_curves(runs, out_dir)

## 7. Test set
Every run is evaluated once on the test set. The detailed analysis uses the run with the best **validation** accuracy.

In [ ]:
best = report.test_summary(runs, test_loader, recipe, out_dir)
scene_acc = report.analyse(best, test_ds, out_dir, title="Attention CNN, trained from scratch")

## 8. Where does the model look?
The learned attention on correct and wrong test images. On the workpiece = the model found it by itself; on the background = a shortcut.

In [ ]:
report.plot_attention(best, test_ds, out_dir)

## 9. Save the outputs to Google Drive
Copied to `MyDrive/thesis_results/attention_cnn/` with the same folders as `outputs/attention_cnn/`:
`weights/`, `figures/`, `results/`. A re-run replaces the files of this model only.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

drive_dir = os.path.join(config.DRIVE_DIR, recipe.model)
!rm -rf "{drive_dir}" && mkdir -p "{drive_dir}" && cp -r {out_dir}/. "{drive_dir}/"
!cd "{drive_dir}" && find . -type f | sort
drive.flush_and_unmount()